In [ ]:
# Fooocus 2.5.5 on Colab (Python 3.13 / NumPy 2 / Starlette 1.x)
import os, re, importlib.util

# 1. Code (latest pygit2 has a Python 3.13 wheel; no libgit2-dev needed)
!pip install -q pygit2
%cd /content
![ -d Fooocus ] || git clone https://github.com/lllyasviel/Fooocus.git
%cd /content/Fooocus
!git checkout -- requirements_versions.txt extras/BLIP/models/med.py

# 2. Unpin packages whose pinned versions don't exist/build on Python 3.13, then install
!sed -i -E 's/^(transformers|tokenizers|safetensors|accelerate|pyyaml|scipy|numpy|onnxruntime|pygit2|rembg)==.*/\1/' requirements_versions.txt
!pip install -q -r requirements_versions.txt "transformers>=4.46,<5"
!pip install -q "numpy>=2.1,<2.3"    # NumPy 2.2.x: new enough for Colab, old enough for numba 0.61

# Fooocus is PyTorch-only; stop transformers from loading Colab's TensorFlow/JAX
os.environ["USE_TF"] = "0"
os.environ["USE_JAX"] = "0"
os.environ["USE_FLAX"] = "0"

# Check: versions, and whether transformers' BERT loads
!python -c "import transformers, torch, numpy; print('transformers', transformers.__version__, '| torch', torch.__version__, '| numpy', numpy.__version__); import transformers.models.bert.modeling_bert; print('BertModel OK')"

# 3. Patch bundled BLIP (image describe) for transformers 4.5x
_p = "/content/Fooocus/extras/BLIP/models/med.py"
_src = open(_p).read()
_src = re.sub(
    r"from transformers\.modeling_utils import",
    '''from transformers.modeling_utils import PreTrainedModel
from transformers import GenerationMixin
from transformers.pytorch_utils import apply_chunking_to_forward, prune_linear_layer
try:
    from transformers.pytorch_utils import find_pruneable_heads_and_indices
except ImportError:
    def find_pruneable_heads_and_indices(heads, n_heads, head_size, already_pruned_heads):
        mask = torch.ones(n_heads, head_size)
        heads = set(heads) - already_pruned_heads
        for head in heads:
            head = head - sum(1 if h < head else 0 for h in already_pruned_heads)
            mask[head] = 0
        mask = mask.view(-1).contiguous().eq(1)
        index = torch.arange(len(mask))[mask].long()
        return heads, index''',
    _src, count=1, flags=re.DOTALL)
_src = _src.replace("class BertLMHeadModel(BertPreTrainedModel):",
                    "class BertLMHeadModel(BertPreTrainedModel, GenerationMixin):")
open(_p, "w").write(_src)

# 4. Patch gradio 3.x for Starlette 1.x (TemplateResponse now takes request first)
_gr = os.path.dirname(importlib.util.find_spec("gradio").origin)
_rp = os.path.join(_gr, "routes.py")
_s = open(_rp).read()
_s = re.sub(r'TemplateResponse\(\s*(\w+),\s*\{"request": request',
            r'TemplateResponse(request, \1, {"request": request', _s)
open(_rp, "w").write(_s)
!grep -n -A1 "TemplateResponse(" {_rp}

# 5. NumPy 1.x compatibility for gradio 3.x and older libraries
import numpy
_site = os.path.dirname(os.path.dirname(numpy.__file__))
open(os.path.join(_site, "numpy2_compat.py"), "w").write('''
import numpy as np

def _obj2sctype(rep, default=None):
    if isinstance(rep, type) and issubclass(rep, np.generic):
        return rep
    try:
        return np.dtype(rep).type
    except Exception:
        return default

_aliases = {
    "float_": np.float64, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp,
    "complex_": np.complex128, "cfloat": np.complex128, "unicode_": np.str_,
    "string_": np.bytes_, "str0": np.str_, "bytes0": np.bytes_, "object0": np.object_,
    "void0": np.void, "longfloat": np.longdouble, "singlecomplex": np.complex64,
    "Inf": np.inf, "NaN": np.nan, "infty": np.inf, "PINF": np.inf, "NINF": -np.inf,
    "obj2sctype": _obj2sctype,
    "issctype": lambda rep: isinstance(rep, type) and issubclass(rep, np.generic),
    "sctype2char": lambda s: np.dtype(s).char,
    "find_common_type": lambda a, s=(): np.result_type(*a, *s),
    "round_": np.round, "product": np.prod, "cumproduct": np.cumprod,
    "sometrue": np.any, "alltrue": np.all,
    "asfarray": lambda a, dtype=np.float64: np.asarray(a, dtype=dtype),
}
for _k, _v in _aliases.items():
    if not hasattr(np, _k):
        setattr(np, _k, _v)
''')
open(os.path.join(_site, "zz_numpy2_compat.pth"), "w").write("import numpy2_compat\n")
!python -c "import numpy as np, supervision; print('numpy shim OK, supervision', supervision.__version__)"

# 6. Launch
!python entry_with_update.py --share --always-high-vram

/content
Cloning into 'Fooocus'...
remote: Enumerating objects: 6735, done.
remote: Total 6735 (delta 0), reused 0 (delta 0), pack-reused 6735 (from 1)
Receiving objects: 100% (6735/6735), 33.35 MiB | 23.98 MiB/s, done.
Resolving deltas: 100% (3850/3850), done.
/content/Fooocus
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 kB 9.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 87.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4